# Hands-on 5: Simpan Data MQTT ke CSV & SQLite

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 3.

Tujuan:
- Menyimpan data MQTT ke CSV (`data/raw/mqtt_room01.csv`).
- Menyimpan data MQTT ke SQLite (`data/raw/iot_sensor.db`).
- Pola multi-storage: satu subscriber menulis ke CSV + SQLite sekaligus.
- (Opsional, butuh server) Menyimpan ke InfluxDB: `scripts/mqtt/save_to_influxdb.py`.

## Tujuan

Pada latihan ini, pembaca akan menyimpan data dari subscriber MQTT ke tiga target berbeda:

- CSV untuk latihan paling sederhana
- SQLite untuk penyimpanan lokal yang ringan
- InfluxDB untuk time-series storage yang lebih cocok untuk data sensor

Dengan latihan ini, pembaca akan memahami bahwa penyimpanan bukan sekadar arsip, tetapi fondasi untuk analitik, visualisasi, dashboard, dan machine learning. Pada arsitektur IoT, data memang perlu diproses, disimpan, lalu dimanfaatkan oleh aplikasi di lapisan yang lebih tinggi.

## 2. Kapan memakai CSV, SQLite, dan InfluxDB?

CSV

Paling cocok untuk: ●   latihan awal ●   ekspor cepat ●   debugging ●   dataset kecil

SQLite

Paling cocok untuk: ●   aplikasi lokal ●   server mini / laptop Ubuntu ●   riwayat data menengah ●   analitik sederhana dengan SQL

InfluxDB

Paling cocok untuk: ●   data sensor time series ●   integrasi dengan Grafana ●   query berbasis waktu ●   dashboard real-time

## 6. Contoh implementasi real untuk penyimpanan

Ruang server

- MQTT menerima suhu dan kelembaban
- SQLite dipakai untuk pencatatan lokal
- InfluxDB dipakai untuk dashboard real-time
- Python ML nanti membaca histori suhu untuk mendeteksi anomali

Greenhouse

- data kelembaban tanah disimpan ke InfluxDB
- Grafana menampilkan tren per jam
- Python subscriber memutuskan kapan pompa menyala

Energi gedung

- data daya listrik dari beberapa ruangan masuk via MQTT
- SQLite dipakai untuk rekap harian
- InfluxDB dipakai untuk tampilan real-time beban gedung

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")


[INFO] Working directory: /downloads/iot-ml-book


In [2]:
import subprocess
import sys
import time

PYTHON = sys.executable
LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def start_process(args, stdout_file):
    f = open(stdout_file, "w", encoding="utf-8")
    proc = subprocess.Popen(args, stdout=f, stderr=subprocess.STDOUT)
    return proc, f

# Bersihkan artefak dari eksekusi sebelumnya agar verifikasi bersih.
for p in ["data/raw/mqtt_room01.csv", "data/raw/iot_sensor.db",
          "data/raw/mqtt_room01_combined.csv", "data/raw/iot_sensor_combined.db"]:
    Path(p).unlink(missing_ok=True)
print("[INFO] Artefak lama dibersihkan.")

[INFO] Artefak lama dibersihkan.


## 1. Jalankan Broker dan Publisher

In [3]:
broker_proc, broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_nb5.log"
)
time.sleep(3)
# Publisher buku apa adanya (interval 5 detik).
pub = latar("scripts/mqtt/publish_sensor.py", "publisher_nb5.log")
time.sleep(1)
print("[INFO] Broker dan publisher berjalan.")

[INFO] Broker dan publisher berjalan.


## 2. Simpan ke CSV

`scripts/mqtt/save_to_csv.py` menulis setiap pesan sebagai baris CSV.

In [4]:
# Skrip buku dijalankan apa adanya (loop_forever); dihentikan setelah 18 detik (3 pesan).
print(jalankan_buku("scripts/mqtt/save_to_csv.py", 18))

Terhubung ke broker.
Data disimpan ke CSV: ['room01', 25.14, 58.55, 553, 116.24, '2026-11-02T12:10:38']
Data disimpan ke CSV: ['room01', 28.99, 60.69, 864, 106.6, '2026-11-02T12:10:43']
Data disimpan ke CSV: ['room01', 24.94, 51.33, 506, 133.06, '2026-11-02T12:10:48']



## 3. Simpan ke SQLite

`scripts/mqtt/save_to_sqlite.py` menulis setiap pesan ke tabel `room_telemetry`.

In [5]:
# Skrip buku dijalankan apa adanya (loop_forever); dihentikan setelah 18 detik (3 pesan).
print(jalankan_buku("scripts/mqtt/save_to_sqlite.py", 18))

Terhubung ke broker.
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 25.26, 'humidity_pct': 50.01, 'co2_ppm': 762, 'power_watt': 157.23, 'timestamp': '2026-11-02T12:10:53'}
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 25.72, 'humidity_pct': 50.35, 'co2_ppm': 533, 'power_watt': 142.05, 'timestamp': '2026-11-02T12:10:58'}
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 27.53, 'humidity_pct': 55.28, 'co2_ppm': 851, 'power_watt': 151.9, 'timestamp': '2026-11-02T12:11:03'}
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 26.33, 'humidity_pct': 48.51, 'co2_ppm': 679, 'power_watt': 138.9, 'timestamp': '2026-11-02T12:11:08'}



## 4. Pola Gabungan: CSV + SQLite Sekaligus

`scripts/mqtt/save_to_csv_sqlite.py` menulis ke dua penyimpanan dalam satu alur.

In [6]:
print(jalankan_buku("scripts/mqtt/save_to_csv_sqlite.py", 18))

# Hentikan publisher dan broker.
henti(pub)
broker_proc.terminate(); broker_f.close()
print("[INFO] Publisher dan broker dihentikan.")

Disimpan ke CSV dan SQLite: ['room01', 24.63, 52.25, 646, 108.4, '2026-11-02T12:11:13']
Disimpan ke CSV dan SQLite: ['room01', 26.25, 50.91, 776, 127.47, '2026-11-02T12:11:18']
Disimpan ke CSV dan SQLite: ['room01', 25.79, 55.45, 880, 114.1, '2026-11-02T12:11:23']

[INFO] Publisher dan broker dihentikan.


## 5. Verifikasi Isi Penyimpanan

In [7]:
import pandas as pd
import sqlite3

df_csv = pd.read_csv("data/raw/mqtt_room01.csv")
print("[INFO] CSV mqtt_room01.csv:", df_csv.shape)
print(df_csv.head())

df_comb = pd.read_csv("data/raw/mqtt_room01_combined.csv")
print("[INFO] CSV mqtt_room01_combined.csv:", df_comb.shape)
print(df_comb.head())

[INFO] CSV mqtt_room01.csv: (3, 6)
  device_id  temperature_c  humidity_pct  co2_ppm  power_watt  \
0    room01          25.14         58.55      553      116.24   
1    room01          28.99         60.69      864      106.60   
2    room01          24.94         51.33      506      133.06   

             timestamp  
0  2026-11-02T12:10:38  
1  2026-11-02T12:10:43  
2  2026-11-02T12:10:48  
[INFO] CSV mqtt_room01_combined.csv: (3, 6)
  device_id  temperature_c  humidity_pct  co2_ppm  power_watt  \
0    room01          24.63         52.25      646      108.40   
1    room01          26.25         50.91      776      127.47   
2    room01          25.79         55.45      880      114.10   

             timestamp  
0  2026-11-02T12:11:13  
1  2026-11-02T12:11:18  
2  2026-11-02T12:11:23  


In [8]:
conn = sqlite3.connect("data/raw/iot_sensor.db")
df_db = pd.read_sql_query("SELECT * FROM room_telemetry", conn)
print("[INFO] SQLite iot_sensor.db:", df_db.shape)
print(df_db.head())
conn.close()

conn = sqlite3.connect("data/raw/iot_sensor_combined.db")
df_dbc = pd.read_sql_query("SELECT * FROM room_telemetry", conn)
print("[INFO] SQLite iot_sensor_combined.db:", df_dbc.shape)
print(df_dbc.head())
conn.close()

[INFO] SQLite iot_sensor.db: (4, 7)
   id device_id  temperature_c  humidity_pct  co2_ppm  power_watt  \
0   1    room01          25.26         50.01      762      157.23   
1   2    room01          25.72         50.35      533      142.05   
2   3    room01          27.53         55.28      851      151.90   
3   4    room01          26.33         48.51      679      138.90   

             timestamp  
0  2026-11-02T12:10:53  
1  2026-11-02T12:10:58  
2  2026-11-02T12:11:03  
3  2026-11-02T12:11:08  
[INFO] SQLite iot_sensor_combined.db: (3, 7)
   id device_id  temperature_c  humidity_pct  co2_ppm  power_watt  \
0   1    room01          24.63         52.25      646      108.40   
1   2    room01          26.25         50.91      776      127.47   
2   3    room01          25.79         55.45      880      114.10   

             timestamp  
0  2026-11-02T12:11:13  
1  2026-11-02T12:11:18  
2  2026-11-02T12:11:23  


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

### 3. Simpan data MQTT ke CSV

Buat file scripts/mqtt/save_to_csv.py:

```python
import csv
import json
from pathlib import Path
import paho.mqtt.client as mqtt
BROKER_HOST = "localhost"
BROKER_PORT = 1883
TOPIC = "kampus/lab1/room01/telemetry"
OUTPUT_FILE = Path("data/raw/mqtt_room01.csv")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
HEADER = [
    "device_id",
    "temperature_c",
    "humidity_pct",
    "co2_ppm",
    "power_watt",
    "timestamp"
]
if not OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as
f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
def on_connect(client, userdata, flags, reason_code,
properties=None):
    print("Terhubung ke broker.")
    client.subscribe(TOPIC, qos=1)
def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode())
         row = [
             data.get("device_id"),
             data.get("temperature_c"),
             data.get("humidity_pct"),
             data.get("co2_ppm"),
             data.get("power_watt"),
             data.get("timestamp"),
         ]
        with open(OUTPUT_FILE, "a", newline="",
encoding="utf-8") as f:
            writer = csv.writer(f)
              writer.writerow(row)
          print(f"Data disimpan ke CSV: {row}")
     except Exception as exc:
         print(f"Gagal menyimpan ke CSV: {exc}")
 client = mqtt.Client()
 client.on_connect = on_connect
 client.on_message = on_message
 client.connect(BROKER_HOST, BROKER_PORT, 60)
 client.loop_forever()
```

Jalankan:

```python
python scripts/mqtt/save_to_csv.py
```

Hasil

Setiap pesan dari MQTT akan ditambahkan ke file: data/raw/mqtt_room01.csv

### 4. Simpan data MQTT ke SQLite

SQLite sangat cocok untuk Ubuntu lokal karena tidak perlu server database terpisah.

Buat file scripts/mqtt/save_to_sqlite.py:

```python
 import json
 import sqlite3
 from pathlib import Path
 import paho.mqtt.client as mqtt
 BROKER_HOST = "localhost"
 BROKER_PORT = 1883
 TOPIC = "kampus/lab1/room01/telemetry"
 DB_PATH = Path("data/raw/iot_sensor.db")
 DB_PATH.parent.mkdir(parents=True, exist_ok=True)
 conn = sqlite3.connect(DB_PATH)
 cursor = conn.cursor()
 cursor.execute("""
 CREATE TABLE IF NOT EXISTS room_telemetry (
     id INTEGER PRIMARY KEY AUTOINCREMENT,
     device_id TEXT,
     temperature_c REAL,
     humidity_pct REAL,
     co2_ppm INTEGER,
     power_watt REAL,
     timestamp TEXT
)
""")
conn.commit()
def on_connect(client, userdata, flags, reason_code,
properties=None):
    print("Terhubung ke broker.")
    client.subscribe(TOPIC, qos=1)
def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode())
        cursor.execute("""
        INSERT INTO room_telemetry (
            device_id, temperature_c, humidity_pct, co2_ppm,
power_watt, timestamp
        ) VALUES (?, ?, ?, ?, ?, ?)
        """, (
            data.get("device_id"),
            data.get("temperature_c"),
            data.get("humidity_pct"),
            data.get("co2_ppm"),
            data.get("power_watt"),
            data.get("timestamp")
        ))
        conn.commit()
         print(f"Data disimpan ke SQLite: {data}")
     except Exception as exc:
         print(f"Gagal menyimpan ke SQLite: {exc}")
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message
client.connect(BROKER_HOST, BROKER_PORT, 60)
try:
    client.loop_forever()
finally:
    conn.close()
```

Cek isi database

```python
Pasang sqlite3 bila belum ada:
       sudo apt install -y sqlite3
```

```python
Lalu cek:
       sqlite3 data/raw/iot_sensor.db
```

```python
Di prompt SQLite:
       .tables
       SELECT * FROM room_telemetry LIMIT 5;
       SELECT AVG(temperature_c) FROM room_telemetry;
       SELECT MAX(co2_ppm) FROM room_telemetry;
       .quit
```

Mengapa SQLite penting?

SQLite memberi pembaca pengalaman nyata memakai query SQL untuk data sensor, tetapi tanpa kompleksitas server DB besar.

### 5. Simpan data MQTT ke InfluxDB

Untuk data sensor berbasis waktu, InfluxDB lebih natural karena dirancang untuk time series. Ini juga sangat cocok untuk integrasi dashboard.

### 5.3. Simpan data ke InfluxDB dengan Python

Buat file scripts/mqtt/save_to_influxdb.py:

import json

```python
 import paho.mqtt.client as mqtt
 from influxdb_client import InfluxDBClient, Point,
 WritePrecision
 from influxdb_client.client.write_api import SYNCHRONOUS
 BROKER_HOST = "localhost"
 BROKER_PORT = 1883
 TOPIC = "kampus/lab1/room01/telemetry"
 INFLUX_URL = "http://localhost:8086"
 INFLUX_TOKEN = "GANTI_DENGAN_TOKEN_ANDA"
 INFLUX_ORG = "iotlab"
 INFLUX_BUCKET = "sensor_data"
 influx_client = InfluxDBClient(
     url=INFLUX_URL,
     token=INFLUX_TOKEN,
     org=INFLUX_ORG
 )
 write_api = influx_client.write_api(write_options=SYNCHRONOUS)
def on_connect(client, userdata, flags, reason_code,
properties=None):
    print("Terhubung ke broker MQTT.")
    client.subscribe(TOPIC, qos=1)
def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode())
        point = (
            Point("room_telemetry")
            .tag("device_id", data.get("device_id", "unknown"))
            .field("temperature_c",
float(data.get("temperature_c", 0)))
            .field("humidity_pct",
float(data.get("humidity_pct", 0)))
            .field("co2_ppm", int(data.get("co2_ppm", 0)))
            .field("power_watt", float(data.get("power_watt",
0)))
            .time(data.get("timestamp"), WritePrecision.S)
        )
        write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG,
record=point)
        print(f"Data disimpan ke InfluxDB: {data}")
     except Exception as exc:
         print(f"Gagal menyimpan ke InfluxDB: {exc}")
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message
client.connect(BROKER_HOST, BROKER_PORT, 60)
try:
    client.loop_forever()
finally:
    influx_client.close()
```

### 7. Pola satu subscriber untuk banyak tujuan

Di sistem nyata, Anda tidak harus memilih hanya satu target. Anda bisa membuat satu subscriber yang:

- menyimpan ke CSV untuk backup mentah
- menyimpan ke SQLite untuk query cepat
- menyimpan ke InfluxDB untuk dashboard

Ini mendukung praktik data acquisition, data storage, visualization, dan preprocessing yang memang menjadi inti IoT analytics.

### 8. Versi gabungan: simpan ke CSV + SQLite sekaligus

Buat file scripts/mqtt/save_to_csv_sqlite.py:

```python
 import csv
 import json
 import sqlite3
 from pathlib import Path
 import paho.mqtt.client as mqtt
 BROKER_HOST = "localhost"
 BROKER_PORT = 1883
 TOPIC = "kampus/lab1/room01/telemetry"
 CSV_PATH = Path("data/raw/mqtt_room01_combined.csv")
DB_PATH = Path("data/raw/iot_sensor_combined.db")
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
if not CSV_PATH.exists():
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "device_id", "temperature_c", "humidity_pct",
            "co2_ppm", "power_watt", "timestamp"
        ])
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS room_telemetry (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    device_id TEXT,
    temperature_c REAL,
    humidity_pct REAL,
    co2_ppm INTEGER,
    power_watt REAL,
    timestamp TEXT
)
""")
conn.commit()
def on_connect(client, userdata, flags, reason_code,
properties=None):
    client.subscribe(TOPIC, qos=1)
def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode())
         row = [
             data.get("device_id"),
             data.get("temperature_c"),
             data.get("humidity_pct"),
             data.get("co2_ppm"),
             data.get("power_watt"),
             data.get("timestamp"),
         ]
         with open(CSV_PATH, "a", newline="", encoding="utf-8")
as f:
              writer = csv.writer(f)
              writer.writerow(row)
         cursor.execute("""
         INSERT INTO room_telemetry (
            device_id, temperature_c, humidity_pct, co2_ppm,
power_watt, timestamp
        ) VALUES (?, ?, ?, ?, ?, ?)
        """, tuple(row))
        conn.commit()
          print(f"Disimpan ke CSV dan SQLite: {row}")
     except Exception as exc:
         print(f"Error: {exc}")
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message
client.connect(BROKER_HOST, BROKER_PORT, 60)
try:
    client.loop_forever()
finally:
    conn.close()
```

### 11. Insight praktis untuk mahasiswa dan praktisi

Dari tiga media penyimpanan ini, pembaca akan belajar satu hal penting: arsitektur data harus proporsional dengan kebutuhan.

- Untuk belajar cepat, CSV sudah cukup.
- Untuk aplikasi lokal yang rapi, SQLite sangat berguna.
- Untuk dashboard dan time-series analytics, InfluxDB jauh lebih nyaman.

Tidak semua proyek harus langsung “besar”. Justru pendekatan bertahap seperti ini lebih cocok untuk laboratorium kampus, bengkel teknik, komunitas maker, dan implementasi awal di lapangan.

## Langkah Kerja dari Buku

### 9. Langkah kerja yang harus dilakukan pembaca

1. Jalankan broker Pastikan Mosquitto aktif.
2. Jalankan publisher Biarkan simulator sensor mengirim data terus-menerus.
3. Pilih target penyimpanan Mulai dari CSV, lalu naik ke SQLite, lalu InfluxDB.
4. Jalankan subscriber penyimpanan Pilih skrip sesuai target.
5. Verifikasi hasil ● buka CSV ● query SQLite ● cek bucket InfluxDB
6. Bandingkan kelebihan Rasakan perbedaan kenyamanan masing-masing pendekatan.

## Tantangan dan Variasi dari Buku

### 10. Tantangan latihan lanjutan

Agar lebih menantang dan inspiratif, pembaca bisa mencoba:

- menyimpan data dari beberapa topic ke satu tabel
- menambahkan kolom location
- membuat tabel SQLite per jenis sensor
- memisahkan bucket InfluxDB untuk room, energy, machine
- menambahkan data kualitas udara lain seperti PM2.5
- menambahkan alert topic jika suhu > 30°C
- membuat skrip pembersihan data mentah menjadi data/processed/

## Solusi Tantangan (Tambahan di Luar Buku)

_Bagian ini BUKAN dari buku. Tantangan pada bagian sebelumnya dikerjakan di sini; kode buku tetap apa adanya._

Catatan untuk pemisahan bucket InfluxDB (room, energy, machine): itu konfigurasi sisi server pada `docker/docker-compose.yml` (bucket terpisah + token), lalu klien menulis dengan parameter `bucket=` yang sesuai. Tidak dijalankan di notebook ini karena membutuhkan stack Docker InfluxDB; lihat README bagian Docker.

In [9]:
# [SOLUSI 1, 2, 5] Satu tabel untuk banyak topik + kolom location + sensor PM2.5.
import json
import sqlite3
import random
from datetime import datetime

import time
import pandas as pd
import paho.mqtt.client as mqtt

sol_broker, sol_broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_solusi_nb5.log")
time.sleep(3)

DB_SOLUSI = "data/raw/iot_sensor_multitopic.db"
Path(DB_SOLUSI).unlink(missing_ok=True)
conn = sqlite3.connect(DB_SOLUSI, check_same_thread=False)
conn.execute("""
CREATE TABLE telemetry_all (
    topic TEXT, device_id TEXT, location TEXT,
    temperature_c REAL, humidity_pct REAL, co2_ppm REAL,
    pm25 REAL, power_watt REAL, timestamp TEXT
)""")

def on_message(client, userdata, msg):
    d = json.loads(msg.payload.decode())
    conn.execute("INSERT INTO telemetry_all VALUES (?,?,?,?,?,?,?,?,?)", (
        msg.topic, d.get("device_id"), d.get("location"),
        d.get("temperature_c"), d.get("humidity_pct"), d.get("co2_ppm"),
        d.get("pm25"), d.get("power_watt"), d.get("timestamp")))
    conn.commit()

sub = mqtt.Client(client_id="solusi-multitopic-sub")
sub.on_message = on_message
sub.connect("localhost", 1883, keepalive=60)
sub.subscribe("kampus/lab1/+/telemetry", qos=1)
sub.loop_start()

pub = mqtt.Client(client_id="solusi-multitopic-pub")
pub.connect("localhost", 1883, keepalive=60)
for topic, dev, loc in [("kampus/lab1/room01/telemetry", "room01", "Lab 1 Lantai 1"),
                        ("kampus/lab1/lab01/telemetry", "lab01", "Lab 2 Lantai 2")]:
    for _ in range(2):
        payload = {"device_id": dev, "location": loc,
                   "temperature_c": round(random.uniform(24, 31), 2),
                   "humidity_pct": round(random.uniform(45, 65), 2),
                   "co2_ppm": random.randint(450, 900),
                   "pm25": round(random.uniform(5, 35), 1),
                   "power_watt": round(random.uniform(100, 160), 2),
                   "timestamp": datetime.now().isoformat(timespec="seconds")}
        pub.publish(topic, json.dumps(payload), qos=1)
        time.sleep(0.3)
pub.disconnect()
time.sleep(2)
sub.loop_stop()
sub.disconnect()

df_all = pd.read_sql_query("SELECT * FROM telemetry_all", conn)
print(f"[SUKSES] Satu tabel menampung {df_all.shape[0]} baris dari {df_all['topic'].nunique()} topik berbeda.")
df_all

[SUKSES] Satu tabel menampung 4 baris dari 2 topik berbeda.


,topic,device_id,location,temperature_c,humidity_pct,co2_ppm,pm25,power_watt,timestamp
0,kampus/lab1/room01/telemetry,room01,Lab 1 Lantai 1,28.04,50.12,755.0,8.4,150.04,2026-11-02T12:11:31
1,kampus/lab1/room01/telemetry,room01,Lab 1 Lantai 1,30.93,62.15,498.0,22.1,137.32,2026-11-02T12:11:32
2,kampus/lab1/lab01/telemetry,lab01,Lab 2 Lantai 2,25.52,55.18,786.0,15.7,104.20,2026-11-02T12:11:32
3,kampus/lab1/lab01/telemetry,lab01,Lab 2 Lantai 2,25.07,51.45,664.0,27.2,119.05,2026-11-02T12:11:32


In [10]:
# [SOLUSI 3] Tabel SQLite terpisah per jenis sensor.
conn.execute("CREATE TABLE room_climate AS SELECT device_id, location, temperature_c, humidity_pct, timestamp FROM telemetry_all")
conn.execute("CREATE TABLE air_quality AS SELECT device_id, location, co2_ppm, pm25, timestamp FROM telemetry_all")
conn.execute("CREATE TABLE energy AS SELECT device_id, location, power_watt, timestamp FROM telemetry_all")
conn.commit()
for t in ["room_climate", "air_quality", "energy"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"[INFO] Tabel {t}: {n} baris")

# [SOLUSI 6] Alert ke topik khusus jika suhu > 30 C.
pub2 = mqtt.Client(client_id="solusi-alert-pub")
pub2.connect("localhost", 1883, keepalive=60)
terkirim = 0
for _, row in df_all.iterrows():
    if row["temperature_c"] and row["temperature_c"] > 30:
        alert = {"device_id": row["device_id"], "location": row["location"],
                 "temperature_c": row["temperature_c"], "alert": "suhu_di_atas_30",
                 "timestamp": row["timestamp"]}
        pub2.publish("kampus/lab1/alert", json.dumps(alert), qos=1)
        terkirim += 1
        print(f"[ALERT] {json.dumps(alert)}")
pub2.disconnect()
print(f"[SUKSES] {terkirim} alert suhu > 30 C dikirim ke kampus/lab1/alert")

sol_broker.terminate(); sol_broker_f.close()
conn.close()
print("[INFO] Broker solusi dihentikan.")

[INFO] Tabel room_climate: 4 baris
[INFO] Tabel air_quality: 4 baris
[INFO] Tabel energy: 4 baris
[ALERT] {"device_id": "room01", "location": "Lab 1 Lantai 1", "temperature_c": 30.93, "alert": "suhu_di_atas_30", "timestamp": "2026-11-02T12:11:32"}
[SUKSES] 1 alert suhu > 30 C dikirim ke kampus/lab1/alert
[INFO] Broker solusi dihentikan.


In [11]:
# [SOLUSI 7] Pembersihan data mentah menjadi data/processed/.
df_mentah = pd.read_csv("data/raw/mqtt_room01_combined.csv")
df_bersih = (df_mentah
             .drop_duplicates()
             .dropna()
             .assign(timestamp=lambda d: pd.to_datetime(d["timestamp"]))
             .sort_values("timestamp"))
out_clean = Path("data/processed/mqtt_room01_clean.csv")
out_clean.parent.mkdir(parents=True, exist_ok=True)
df_bersih.to_csv(out_clean, index=False)
print(f"[SUKSES] {df_mentah.shape[0]} baris mentah -> {df_bersih.shape[0]} baris bersih di {out_clean}")

[SUKSES] 3 baris mentah -> 3 baris bersih di data/processed/mqtt_room01_clean.csv


## Penutup dari Buku

Dengan menyelesaikan latihan ini, pembaca telah membangun satu fondasi IoT yang utuh: publisher mengirim data ke MQTT, subscriber Python menerima data, lalu data disimpan untuk analitik dan visualisasi. Ini adalah bentuk paling nyata dari alur sensor → broker → storage, yang nanti akan diperluas menjadi analytics → model → aksi pada bab-bab berikutnya. Pendekatan ini juga sepenuhnya selaras dengan semangat buku: praktis, open source, mudah direplikasi di Ubuntu, dan cukup kuat untuk berkembang menjadi pipeline IoT-ML lengkap.

Pada tahap berikutnya, materi ini sudah sangat siap untuk dilanjutkan ke Bab 3, khususnya:

- eksplorasi dataset sensor dengan pandas,
- pembersihan timestamp,
- resampling,
- imputasi,
- dan deteksi anomali dasar.

## Penutup

Data telemetri MQTT kini tersimpan ke dua bentuk penyimpanan lokal:
CSV (mudah dibaca pandas) dan SQLite (query relasional). Untuk kebutuhan
time-series skala besar, buku menyediakan jalur InfluxDB
(`scripts/mqtt/save_to_influxdb.py`) yang memerlukan server InfluxDB
(Docker `influxdb:2.7`) dan pustaka `influxdb-client`.
Dataset tersimpan ini menjadi bahan eksplorasi pada Hands-on 6.